<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/AfriBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# AFRIBERTA Model
# ==========================================================

!pip install -U transformers datasets accelerate scikit-learn -q

# =====================================
# 1. Imports
# =====================================
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# =====================================
# 2. LOAD DATA SPLITS
# =====================================

train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')
test_df  = pd.read_csv('/content/test.csv')

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

# =====================================
# 3. Convert to HuggingFace Dataset
# =====================================

train_dataset = Dataset.from_pandas(train_df[["Text", "label"]])
val_dataset   = Dataset.from_pandas(val_df[["Text", "label"]])
test_dataset  = Dataset.from_pandas(test_df[["Text", "label"]])

# =====================================
# 4. Load AfriBERTa
# =====================================

model_name = "castorini/afriberta_base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# =====================================
# 5. Tokenization
# =====================================

def tokenize_function(example):
    return tokenizer(
        example["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# =====================================
# 6. Metrics
# =====================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# =====================================
# 7. Training Arguments
# =====================================

training_args = TrainingArguments(
    output_dir="./afriberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.02,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=200,
    save_total_limit=2,
    seed=42
)

# =====================================
# 8. Trainer
# =====================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# =====================================
# 9. Train
# =====================================

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

trainer.train()

# =====================================
# 10. Evaluate TEST SET
# =====================================

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(test_df["label"], preds)

print("\n==============================")
print("AFRIBERTA RESULTS (ALIGNED)")
print("==============================")
print("Test Size:", len(test_df))
print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:\n")
print(classification_report(
    test_df["label"],
    preds,
    target_names=["Negative", "Neutral", "Positive"]
))

# =====================================
# 11. Confusion Matrix
# =====================================

cm = confusion_matrix(test_df["label"], preds)

ConfusionMatrixDisplay(
    cm,
    display_labels=["Negative", "Neutral", "Positive"]
).plot(cmap="Blues")

plt.title("AfriBERTa Confusion Matrix (Aligned)")
plt.show()

# =====================================
# 12. SAVE PREDICTIONS (for McNemar test)
# =====================================

pd.DataFrame({
    "y_true": test_df["label"].values,
    "afri_preds": preds
}).to_csv("/content/afri_preds.csv", index=False)

print("\nPredictions saved for statistical testing")